In [1]:
# ============================================================
# Cell 1: Setup
# 16_naked_atm_put_incremental_update_and_intraday_v0_1
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

cwd = Path.cwd()

if cwd.name.lower() == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"
PRODUCTION_DIR = DATA_DIR / "production"

THETADATA_CHAIN_DIR = RAW_DATA_DIR / "thetadata_chains"

for d in [RAW_DATA_DIR, PROCESSED_DATA_DIR, AUDIT_DIR, PRODUCTION_DIR, THETADATA_CHAIN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 250)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("THETADATA_CHAIN_DIR:", THETADATA_CHAIN_DIR)

PROJECT_ROOT: C:\Users\patri\vrp_project
THETADATA_CHAIN_DIR: C:\Users\patri\vrp_project\data\raw\thetadata_chains


In [2]:
# ============================================================
# Cell 2: Production file paths and constants
# ============================================================

NAKED_ATM_EOD_PANEL = (
    PROCESSED_DATA_DIR / "naked_atm_put_eod_panel_v0_1.parquet"
)

NAKED_ATM_INTRADAY_SNAPSHOT = (
    PROCESSED_DATA_DIR / "naked_atm_put_intraday_snapshot_v0_1.parquet"
)

NAKED_ATM_SIGNAL_PANEL = (
    PROCESSED_DATA_DIR / "naked_atm_put_production_signal_panel_v0_1.parquet"
)

TARGET_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

assert NAKED_ATM_EOD_PANEL.exists(), NAKED_ATM_EOD_PANEL

print("EOD panel:", NAKED_ATM_EOD_PANEL)
print("Intraday snapshot:", NAKED_ATM_INTRADAY_SNAPSHOT)
print("Signal panel:", NAKED_ATM_SIGNAL_PANEL)
print("Target tenors:", TARGET_TENORS)

EOD panel: C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet
Intraday snapshot: C:\Users\patri\vrp_project\data\processed\naked_atm_put_intraday_snapshot_v0_1.parquet
Signal panel: C:\Users\patri\vrp_project\data\processed\naked_atm_put_production_signal_panel_v0_1.parquet
Target tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]


In [3]:
# ============================================================
# Cell 3: Load current EOD panel
# ============================================================

eod_panel = pd.read_parquet(NAKED_ATM_EOD_PANEL).copy()

eod_panel["trade_dt"] = pd.to_datetime(eod_panel["trade_dt"]).dt.normalize()
eod_panel["entry_tenor"] = eod_panel["entry_tenor"].astype(int)

latest_panel_date = eod_panel["trade_dt"].max()

print("Current EOD panel:")
print("Rows:", len(eod_panel))
print("Start:", eod_panel["trade_dt"].min().date())
print("End:", latest_panel_date.date())
print("Unique dates:", eod_panel["trade_dt"].nunique())
print("Unique tenors:", sorted(eod_panel["entry_tenor"].unique().tolist()))

display(eod_panel.tail(9))

Current EOD panel:
Rows: 18099
Start: 2018-06-25
End: 2026-06-25
Unique dates: 2011
Unique tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]


,trade_dt,quote_timestamp,entry_tenor,expiry,actual_dte,underlying_price,atm_strike,atm_put_bid,atm_put_ask,atm_put_mid,atm_put_iv,atm_put_delta,tenor_group,vix_style_vol,implied_variance,spx_rsi_14,entry_rsi_14,old_vrp_signal,old_vrp_3m_z,old_vrp_1y_z,old_vrp_blended_z,simple_carry_vrp_signal,simple_carry_vrp_signal_z_3m,simple_carry_vrp_signal_z_1y,simple_carry_vrp_min_z,trailing_carry_vrp_signal,trailing_carry_vrp_signal_z_3m,trailing_carry_vrp_signal_z_1y,trailing_carry_vrp_min_z,expiry_spx_close,expiry_intrinsic_value,expired_otm,entry_credit_points,short_option_pnl_points,normalized_pnl_dollars,test_pnl,win,test_win,pricing_status,mapping_status,selected_chain_file,cache_status,selected_chain_file_live,put_moneyness,put_pct_otm
18090,2026-06-25,2026-06-25T16:00:00.000,9,2026-07-02,7,7357.49,7355.0,59.9,61.5,60.70,NaN,NaN,front_9_15d,17.952376,0.032229,46.297996,46.297996,0.483909,0.114250,-0.395434,-0.140592,0.773380,0.442191,-0.190450,-0.190450,0.483909,0.128162,-0.395629,-0.395629,NaN,NaN,None,59.9,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260702_160000.pkl,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260702_160000.pkl,0.999662,0.000338
18091,2026-06-25,2026-06-25T16:00:00.000,12,2026-07-10,15,7357.49,7355.0,84.8,86.2,85.50,NaN,NaN,front_9_15d,17.964763,0.032273,46.297996,46.297996,0.293640,-0.378609,-0.842333,-0.610471,0.683950,0.156820,-0.507614,-0.507614,0.293640,-0.359177,-0.840252,-0.840252,NaN,NaN,None,84.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,0.999662,0.000338
18092,2026-06-25,2026-06-25T16:00:00.000,15,2026-07-10,15,7357.49,7355.0,84.8,86.2,85.50,NaN,NaN,front_9_15d,17.972192,0.032300,46.297996,46.297996,0.171854,-0.531361,-1.088037,-0.809699,0.509373,-0.115243,-0.801644,-0.801644,0.171854,-0.509206,-1.082160,-1.082160,NaN,NaN,None,84.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,0.999662,0.000338
18093,2026-06-25,2026-06-25T16:00:00.000,18,2026-07-10,15,7357.49,7355.0,84.8,86.2,85.50,NaN,NaN,middle_18_24d,17.707876,0.031357,46.297996,46.297996,0.099408,-0.773276,-1.333368,-1.053322,0.540072,-0.146100,-0.843356,-0.843356,0.099408,-0.747389,-1.323083,-1.323083,NaN,NaN,None,84.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,0.999662,0.000338
18094,2026-06-25,2026-06-25T16:00:00.000,21,2026-07-17,22,7357.49,7355.0,100.8,101.9,101.35,NaN,NaN,middle_18_24d,17.516638,0.030683,46.297996,46.297996,-0.182296,-1.505797,-1.955464,-1.730630,0.277368,-0.476921,-1.148563,-1.148563,-0.182296,-1.456025,-1.932966,-1.932966,NaN,NaN,None,100.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl,0.999662,0.000338
18095,2026-06-25,2026-06-25T16:00:00.000,24,2026-07-17,22,7357.49,7355.0,100.8,101.9,101.35,NaN,NaN,middle_18_24d,18.028765,0.032504,46.297996,46.297996,-0.024917,-1.193720,-1.737542,-1.465631,0.399541,-0.201362,-0.928478,-0.928478,-0.024917,-1.158431,-1.719697,-1.719697,NaN,NaN,None,100.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl,0.999662,0.000338
18096,2026-06-25,2026-06-25T16:00:00.000,27,2026-07-24,29,7357.49,7350.0,115.6,116.8,116.20,NaN,NaN,back_27_33

In [6]:
# ============================================================
# Cell 4: Dynamic EOD backfill window and intraday run date
# ============================================================

from datetime import time
from zoneinfo import ZoneInfo

# Normal daily use:
#   AUTO mode figures out the latest completed EOD date.
#
# Logic:
#   - Before EOD data is available: latest completed EOD = prior trading day
#   - After EOD data is available: latest completed EOD = today, if today is a trading day
#
# Adjust this if ThetaData EOD files are reliably ready earlier/later.
EOD_AVAILABLE_AFTER_ET = time(18, 30)

NY_TZ = ZoneInfo("America/New_York")
now_et = pd.Timestamp.now(tz=NY_TZ)

today_et = now_et.normalize().tz_localize(None)
current_time_et = now_et.time()

# Use NYSE calendar if available. Fall back to business days.
def get_trading_days(start_date, end_date):
    start_date = pd.Timestamp(start_date).normalize()
    end_date = pd.Timestamp(end_date).normalize()

    try:
        import pandas_market_calendars as mcal

        nyse = mcal.get_calendar("NYSE")
        schedule = nyse.schedule(
            start_date=start_date.strftime("%Y-%m-%d"),
            end_date=end_date.strftime("%Y-%m-%d"),
        )

        sessions = (
            pd.Series(schedule.index)
            .dt.tz_localize(None)
            .dt.normalize()
            .sort_values()
            .reset_index(drop=True)
        )

        calendar_source = "pandas_market_calendars_NYSE"

    except Exception:
        sessions = (
            pd.Series(pd.bdate_range(start_date, end_date))
            .dt.normalize()
            .sort_values()
            .reset_index(drop=True)
        )

        calendar_source = "pandas_bdate_range_fallback"

    return sessions, calendar_source


# Reload current EOD panel.
eod_panel = pd.read_parquet(NAKED_ATM_EOD_PANEL).copy()
eod_panel["trade_dt"] = pd.to_datetime(eod_panel["trade_dt"]).dt.normalize()
eod_panel["entry_tenor"] = eod_panel["entry_tenor"].astype(int)

latest_panel_date = eod_panel["trade_dt"].max()

# Build trading calendar from the day after the current panel through today.
future_trading_days, calendar_source = get_trading_days(
    latest_panel_date + pd.Timedelta(days=1),
    today_et,
)

if len(future_trading_days) == 0:
    latest_completed_eod_date = latest_panel_date
else:
    today_is_trading_day = today_et in set(future_trading_days)

    if today_is_trading_day and current_time_et >= EOD_AVAILABLE_AFTER_ET:
        latest_completed_eod_date = today_et
    else:
        prior_sessions = future_trading_days[future_trading_days < today_et]

        if len(prior_sessions) > 0:
            latest_completed_eod_date = prior_sessions.max()
        else:
            latest_completed_eod_date = latest_panel_date

intraday_run_date = today_et

# EOD dates to permanently add to history.
eod_update_dates = future_trading_days[
    (future_trading_days > latest_panel_date)
    & (future_trading_days <= latest_completed_eod_date)
]

eod_update_dates_df = pd.DataFrame({
    "eod_update_date": eod_update_dates,
})

print("Now ET:", now_et)
print("Calendar source:", calendar_source)
print("Latest panel date:", latest_panel_date.date())
print("Latest completed EOD date:", latest_completed_eod_date.date())
print("Intraday run date:", intraday_run_date.date())
print("EOD dates to add:", len(eod_update_dates_df))

display(eod_update_dates_df)

Now ET: 2026-07-01 20:45:29.746038-04:00
Calendar source: pandas_market_calendars_NYSE
Latest panel date: 2026-06-25
Latest completed EOD date: 2026-07-01
Intraday run date: 2026-07-01
EOD dates to add: 4


,eod_update_date
0,2026-06-26
1,2026-06-29
2,2026-06-30
3,2026-07-01


In [7]:
# ============================================================
# Cell 5: Check existing ThetaData cache for EOD update dates
# ============================================================

chain_files = sorted(THETADATA_CHAIN_DIR.glob("*.pkl"))

cache_rows = []

for path in chain_files:
    parts = path.stem.split("_")

    if len(parts) < 4:
        continue

    root = parts[0]
    trade_date_raw = parts[1]
    expiry_raw = parts[2]
    quote_time_raw = parts[3]

    trade_dt = pd.to_datetime(trade_date_raw, format="%Y%m%d", errors="coerce")
    expiry = pd.to_datetime(expiry_raw, format="%Y%m%d", errors="coerce")

    cache_rows.append({
        "root": root,
        "trade_dt": trade_dt,
        "expiry": expiry,
        "quote_time": quote_time_raw,
        "file": path.name,
        "path": str(path),
        "size_mb": path.stat().st_size / 1_000_000,
    })

chain_cache_inventory = pd.DataFrame(cache_rows)

update_dates_set = set(pd.to_datetime(eod_update_dates_df["eod_update_date"]).dt.normalize())

update_date_cache = chain_cache_inventory[
    chain_cache_inventory["trade_dt"].isin(update_dates_set)
].copy()

print("Total cached chain files:", len(chain_cache_inventory))
print("Cached chain date range:", chain_cache_inventory["trade_dt"].min(), "to", chain_cache_inventory["trade_dt"].max())
print("EOD update dates needed:", len(update_dates_set))
print("Cached files for update dates:", len(update_date_cache))

if len(update_date_cache) > 0:
    cache_summary = (
        update_date_cache
        .groupby("trade_dt")
        .agg(
            chain_files=("file", "count"),
            expiries=("expiry", lambda x: sorted(pd.to_datetime(x).dt.date.unique().tolist())),
            roots=("root", lambda x: sorted(set(x))),
            quote_times=("quote_time", lambda x: sorted(set(x))),
        )
        .reset_index()
    )

    display(cache_summary)
    display(update_date_cache.sort_values(["trade_dt", "expiry", "root", "quote_time"]))
else:
    print("No cached chain files found for the EOD update dates.")

Total cached chain files: 10901
Cached chain date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
EOD update dates needed: 4
Cached files for update dates: 0
No cached chain files found for the EOD update dates.


In [8]:
# ============================================================
# Cell 6: Discover local ThetaData interface
# ============================================================

import sys
import inspect
import requests
import importlib.util

print("Python:", sys.executable)

# 1. Check whether the thetadata package is installed.
theta_spec = importlib.util.find_spec("thetadata")
print("thetadata package found:", theta_spec is not None)

if theta_spec is not None:
    import thetadata
    print("thetadata module:", thetadata)

    public_names = [x for x in dir(thetadata) if not x.startswith("_")]
    print("\nPublic thetadata names:")
    print(public_names)

    if hasattr(thetadata, "ThetaClient"):
        print("\nThetaClient found.")
        ThetaClient = thetadata.ThetaClient

        methods = [
            x for x in dir(ThetaClient)
            if not x.startswith("_")
        ]

        print("\nThetaClient public methods:")
        print(methods)

# 2. Check local ThetaData ports we have discussed before.
ports_to_check = [25510, 25503]

for port in ports_to_check:
    url = f"http://127.0.0.1:{port}"

    try:
        r = requests.get(url, timeout=2)
        print(f"\n{url}: status={r.status_code}")
        print(r.text[:500])
    except Exception as e:
        print(f"\n{url}: not reachable / no root response")
        print(type(e).__name__, str(e)[:300])

Python: C:\Users\patri\AppData\Local\Programs\Python\Python313\python.exe
thetadata package found: True
thetadata module: <module 'thetadata' from 'C:\\Users\\patri\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\thetadata\\__init__.py'>

Public thetadata names:
['Any', 'Contract', 'DataFrame', 'DataType', 'DateRange', 'Exchange', 'HEADER_FIELDS', 'HEADER_MAX_LENGTH', 'Header', 'ListBody', 'MessageType', 'NoData', 'OptionReqType', 'OptionRight', 'Optional', 'Quote', 'QuoteCondition', 'ReconnectingToServer', 'ResponseError', 'ResponseParseError', 'SecType', 'Series', 'StockReqType', 'StreamMsg', 'StreamMsgType', 'StreamResponseType', 'ThetaClient', 'TickBody', 'Trade', 'TradeCondition', 'annotations', 'client', 'dataclass', 'date', 'datetime', 'enum', 'enums', 'exceptions', 'ijson', 'json', 'np', 'parse_flexible_REST', 'parse_header_REST', 'parse_hist_REST', 'parse_hist_REST_stream', 'parse_hist_REST_stream_ijson', 'parse_list_REST', 'parsing', 'pd', 'requests', 'termi

In [9]:
# ============================================================
# Cell 7: Inspect ThetaClient REST method signatures
# ============================================================

from thetadata import ThetaClient
import inspect

methods_to_inspect = [
    "get_expirations_REST",
    "get_strikes_REST",
    "get_opt_at_time_REST",
    "get_stk_at_time_REST",
]

for method_name in methods_to_inspect:
    method = getattr(ThetaClient, method_name)

    print("=" * 80)
    print(method_name)
    print("Signature:")
    print(inspect.signature(method))

    doc = inspect.getdoc(method)
    print("\nDocstring:")
    if doc is None:
        print("None")
    else:
        print(doc[:2000])

get_expirations_REST
Signature:
(self, root: str, host: str = '127.0.0.1', port: str = '25510') -> pandas.core.series.Series

Docstring:
Get all options expirations for a provided underlying root.

:param root:           The root / underlying / ticker / symbol.
:param host:           The ip address of the server
:param port:           The port of the server

:return:               All expirations that ThetaData provides data for.
:raises ResponseError: If the request failed.
:raises NoData:        If there is no data available for the request.
get_strikes_REST
Signature:
(self, root: str, exp: datetime.date, date_range: thetadata.enums.DateRange = None, host: str = '127.0.0.1', port: str = '25510') -> pandas.core.series.Series

Docstring:
Get all options strike prices in US tenths of a cent.

:param root:           The root / underlying / ticker / symbol.
:param exp:            The expiration date.
:param date_range:     If specified, this function will return strikes only if they have

In [11]:
# ============================================================
# Cell 8: Test direct ThetaData v3 REST calls
# ============================================================

import requests
import pandas as pd
from io import StringIO

THETA_BASE_URL = "http://127.0.0.1:25503"

def theta_v3_get(endpoint, params):
    url = f"{THETA_BASE_URL}{endpoint}"
    params = dict(params)
    params["format"] = "json"

    r = requests.get(url, params=params, timeout=60)

    print("URL:", r.url)
    print("Status:", r.status_code)

    if r.status_code != 200:
        print(r.text[:1000])
        r.raise_for_status()

    data = r.json()

    if isinstance(data, list):
        return pd.DataFrame(data)

    if isinstance(data, dict):
        # Some endpoints may wrap data.
        for key in ["data", "response", "result"]:
            if key in data and isinstance(data[key], list):
                return pd.DataFrame(data[key])
        return pd.DataFrame([data])

    return pd.DataFrame(data)


# Test 1: list strikes for SPXW 2026-07-02
test_strikes = theta_v3_get(
    "/v3/option/list/strikes",
    {
        "symbol": "SPXW",
        "expiration": "20260702",
    },
)

print("Strike rows:", len(test_strikes))
display(test_strikes.head())
display(test_strikes.tail())


# Test 2: get 16:00 quote for all put strikes near spot on one expiry/date
test_quote = theta_v3_get(
    "/v3/option/at_time/quote",
    {
        "symbol": "SPXW",
        "expiration": "20260702",
        "start_date": "20260626",
        "end_date": "20260626",
        "time_of_day": "16:00:00.000",
        "right": "put",
        "strike": "*",
        "strike_range": 20,
    },
)

print("Quote rows:", len(test_quote))
print(test_quote.columns.tolist())
display(test_quote.head())
display(test_quote.tail())

URL: http://127.0.0.1:25503/v3/option/list/strikes?symbol=SPXW&expiration=20260702&format=json
Status: 200
Strike rows: 1


,symbol,strike
0,"[SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW,...","[4200.0, 8400.0, 7590.0, 7185.0, 6780.0, 7355.0, 7910.0, 7100.0, 7675.0, 7420.0, 7035.0, 6630.0, 7740.0, 5800.0, 6950.0, 7505.0, 7270.0, 7825.0, 7975.0, 7570.0, 6075.0, 7165.0, 6760.0, 7890.0, 6525.0, 7080.0, 7655.0, 3200.0, 7400.0, 8125.0, 5500...."


,symbol,strike
0,"[SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW,...","[4200.0, 8400.0, 7590.0, 7185.0, 6780.0, 7355.0, 7910.0, 7100.0, 7675.0, 7420.0, 7035.0, 6630.0, 7740.0, 5800.0, 6950.0, 7505.0, 7270.0, 7825.0, 7975.0, 7570.0, 6075.0, 7165.0, 6760.0, 7890.0, 6525.0, 7080.0, 7655.0, 3200.0, 7400.0, 8125.0, 5500...."


URL: http://127.0.0.1:25503/v3/option/at_time/quote?symbol=SPXW&expiration=20260702&start_date=20260626&end_date=20260626&time_of_day=16%3A00%3A00.000&right=put&strike=%2A&strike_range=20&format=json
Status: 200
Quote rows: 1
['symbol', 'ask_size', 'ask_condition', 'strike', 'right', 'bid_size', 'ask_exchange', 'bid_exchange', 'ask', 'expiration', 'bid', 'bid_condition', 'timestamp']


,symbol,ask_size,ask_condition,strike,right,bid_size,ask_exchange,bid_exchange,ask,expiration,bid,bid_condition,timestamp
0,"[SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW]","[1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 15, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1]","[50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50]","[7250.0, 7335.0, 7270.0, 7355.0, 7380.0, 7295.0, 7230.0, 7400.0, 7315.0, 7405.0, 7340.0, 7255.0, 7360.0, 7275.0, 7365.0, 7300.0, 7215.0, 7385.0, 7320.0, 7235.0, 7410.0, 7325.0, 7260.0, 7345.0, 7280.0, 7370.0, 7285.0, 7220.0, 7390.0, 7305.0, 7240....","[PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT]","[1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 2, 1, 5, 1, 1, 1, 1, 1, 1, 3, 1, 2, 1, 1, 1, 1, 1, 1]","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]","[42.0, 84.0, 53.0, 101.0, 110.0, 56.0, 42.0, 121.0, 67.4, 123.0, 91.0, 47.0, 103.0, 52.0, 104.0, 59.0, 28.0, 111.0, 86.0, 45.0, 128.0, 86.0, 47.9, 93.0, 55.0, 110.0, 57.0, 40.0, 116.0, 83.0, 46.0, 86.3, 45.0, 95.0, 50.0, 56.0, 109.0, 82.0, 41.0, ...","[2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-...","[10.0, 13.0, 15.0, 23.0, 60.0, 51.2, 4.0, 43.0, 59.6, 45.0, 13.0, 9.0, 25.0, 14.0, 26.0, 33.5, 27.4, 35.0, 8.0, 7.0, 50.0, 8.0, 39.8, 15.0, 45.9, 32.0, 19.0, 2.0, 38.0, 5.0, 8.0, 49.7, 7.0, 40.2, 12.0, 18.0, 31.0, 4.0, 3.0, 37.0]","[50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50]","[2026-06-26T15:59:59.963, 2026-06-26T15:59:59.823, 2026-06-26T15:59:59.959, 2026-06-26T15:59:59.825, 2026-06-26T15:59:59.802, 2026-06-26T15:59:59.985, 2026-06-26T15:59:59.963, 2026-06-26T15:59:59.803, 2026-06-26T15:59:59.995, 2026-06-26T15:59:59...."


,symbol,ask_size,ask_condition,strike,right,bid_size,ask_exchange,bid_exchange,ask,expiration,bid,bid_condition,timestamp
0,"[SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW, SPXW]","[1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 15, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1]","[50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50]","[7250.0, 7335.0, 7270.0, 7355.0, 7380.0, 7295.0, 7230.0, 7400.0, 7315.0, 7405.0, 7340.0, 7255.0, 7360.0, 7275.0, 7365.0, 7300.0, 7215.0, 7385.0, 7320.0, 7235.0, 7410.0, 7325.0, 7260.0, 7345.0, 7280.0, 7370.0, 7285.0, 7220.0, 7390.0, 7305.0, 7240....","[PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT, PUT]","[1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 2, 1, 5, 1, 1, 1, 1, 1, 1, 3, 1, 2, 1, 1, 1, 1, 1, 1]","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]","[42.0, 84.0, 53.0, 101.0, 110.0, 56.0, 42.0, 121.0, 67.4, 123.0, 91.0, 47.0, 103.0, 52.0, 104.0, 59.0, 28.0, 111.0, 86.0, 45.0, 128.0, 86.0, 47.9, 93.0, 55.0, 110.0, 57.0, 40.0, 116.0, 83.0, 46.0, 86.3, 45.0, 95.0, 50.0, 56.0, 109.0, 82.0, 41.0, ...","[2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-07-02, 2026-...","[10.0, 13.0, 15.0, 23.0, 60.0, 51.2, 4.0, 43.0, 59.6, 45.0, 13.0, 9.0, 25.0, 14.0, 26.0, 33.5, 27.4, 35.0, 8.0, 7.0, 50.0, 8.0, 39.8, 15.0, 45.9, 32.0, 19.0, 2.0, 38.0, 5.0, 8.0, 49.7, 7.0, 40.2, 12.0, 18.0, 31.0, 4.0, 3.0, 37.0]","[50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50]","[2026-06-26T15:59:59.963, 2026-06-26T15:59:59.823, 2026-06-26T15:59:59.959, 2026-06-26T15:59:59.825, 2026-06-26T15:59:59.802, 2026-06-26T15:59:59.985, 2026-06-26T15:59:59.963, 2026-06-26T15:59:59.803, 2026-06-26T15:59:59.995, 2026-06-26T15:59:59...."
